# Анализ ТВ-шоу — Даниил

### Даниил. Анализ ТВ-шоу

В этой части посмотрим конкретно на сериалы. Нашей задачей будет разобраться с динамикой выпусков, разбить контент по сегментам аудитории и оценить, как пользователи реагируют на разные категории.

**Как готовили данные:**
Из общей кучи взяли только телевизионный контент (`type == 'tv'`). Пропуски в колонке `user rating score` просто выкинули через `dropna()`. Специально не стали заполнять их средним или медианой, чтобы не размывать реальную картину и не портить распределение оценок. После очистки осталось ровно 171 строка, с этим объемом и работаем.

Что конкретно сделано ниже:
* Отфильтровали датасет и убрали пустые оценки.
* Разобрали динамику по годам и нашли причину аномального скачка в 2016 году.
* Свели мелкие рейтинги в крупные сегменты аудитории и проверили оценки по ним.

In [1]:
# Импортнем нужные библиотеки
import pandas as pd
import plotly.express as px

In [2]:
# Еще раз грузим данные
df_raw = pd.read_csv("NetflixShows_clear.csv")

# Отфильтруем только ТВ-шоу и сразу удаляем строки, где нет оценок
df_tv = df_raw[df_raw['type'] == 'tv'].dropna(subset=['user rating score'])

# Оставляем только нужные для анализа колонки
columns_to_keep = ['title', 'rating', 'release year', 'user rating score', 'audience_segment']
df_tv = df_tv[columns_to_keep]

# Посмотрим на размер очищенного датасета и первые его несколько строк
print(f"Размер очищенного датасета по ТВ-шоу: {df_tv.shape}")
df_tv.head()

Размер очищенного датасета по ТВ-шоу: (171, 5)


,title,rating,release year,user rating score,audience_segment
2,Grey's Anatomy,TV-14,2016,98.0,Teens
3,Prison Break,TV-14,2008,98.0,Teens
4,How I Met Your Mother,TV-PG,2014,94.0,Family
5,Supernatural,TV-14,2016,95.0,Teens
6,Breaking Bad,TV-MA,2013,97.0,Adults


#### Динамика выпуска ТВ-шоу по годам

Теперь будем агрегировать данные, чтобы посмотреть, сколько сериалов из нашего датасета выпускалось в разные годы, а также рассчитаем их средний рейтинг.

In [3]:
# Сгруппируем по годам и посчитаем количество сериалов и среднюю оценку
yearly_stats = df_tv.groupby('release year')['user rating score'].agg(['count', 'mean']).reset_index()
yearly_stats.columns = ['Год выпуска', 'Количество шоу', 'Средний рейтинг']

# Округляем рейтинг
yearly_stats['Средний рейтинг'] = yearly_stats['Средний рейтинг'].round(2)

display(yearly_stats.sort_values(by='Год выпуска', ascending=False))

,Год выпуска,Количество шоу,Средний рейтинг
17,2017,16,88.12
16,2016,68,82.81
15,2015,25,82.12
14,2014,10,77.20
13,2013,9,80.78
12,2012,13,82.00
11,2011,2,89.00
10,2010,6,78.33
9,2009,1,74.00
8,2008,3,80.67


**Промежуточный вывод:**

По графикам сразу виден мощный скачок в 2016 году. В нашей выборке на этот год приходится сразу 68 сериалов, а это почти 40% от всех ТВ-шоу в датасете и практически в три раза больше, чем в 2015-м (там было 25).

Этот всплеск легко объясняется исторически: как раз в январе 2016 года Netflix запустился сразу в 190 странах. Платформе нужно было резко забивать библиотеку контентом, чтобы удерживать новых подписчиков по всему миру. При этом средняя оценка удержалась на уровне 82.81 балла, то есть качество из-за объемов не просело. А уже в 2017 году количество релизов в выборке снизилось до 16, но средний балл долетел до пиковых 88.13.

#### Распределение объёма контента по сегментам аудитории

Теперь посмотрим, на какую именно аудиторию ориентированы сериалы и посчитаем количество проектов по укрупненным сегментам (audience_segment).

In [4]:
# Посчитаем количество шоу по укрупненным сегментам аудитории
# (audience_segment рассчитан на этапе обработки данных в ноутбуке 01)
segment_counts = df_tv['audience_segment'].value_counts().reset_index()

# Переименуем колонки для наглядности
segment_counts.columns = ['Сегмент аудитории', 'Количество сериалов']

display(segment_counts)

,Сегмент аудитории,Количество сериалов
0,Teens,77
1,Adults,40
2,Kids,33
3,Family,21


In [5]:
# Теперь построим простой график с помощью модуля express из библиотеки plotly (https://plotly.com/python/bar-charts/)
fig_volume = px.bar(segment_counts,
                    x='Сегмент аудитории',
                    y='Количество сериалов',
                    title='Количество выпущенных ТВ-шоу по сегментам аудитории',
                    color='Сегмент аудитории')

fig_volume.show()

**Промежуточный вывод:**

Наша разбивка по сегментам наглядно показывает, как устроен продакшн Netflix:

1. Главный фокус (Ядро аудитории): больше всего контента идет в категорию Teens (77 шоу) и Adults (40 шоу). Вместе они забирают 68.4% (117 из 171 шоу) всей выборки. Вполне очевидно, что Netflix целится в подростков постарше и взрослых, которые являются самой активной и платящей аудиторией, которая генерирует просмотры.
2. Детские проекты: Сегмент Kids идет на третьем месте (33 шоу). Сюда упали все мелкие детские рейтинги после маппинга.
3. Семейный контент: Меньше всего шоу получилось в категории Family (21 проект). Это такой безопасный компромисс, чтобы посмотреть что-то всей семьей.

По итогу: в своих сериалах Netflix явно делает ставку на взрослый и подростковый контент с закрученным сюжетом, а детские и семейные шоу выпускает скорее как сопутствующие.

#### Анализ оценок пользователей по сегментам аудитории

Проверим, шоу для каких сегментов аудитории залетают пользователям лучше всего и получают более высокие оценки.

In [6]:
# Посчитаем медианные оценки по нашим укрупнённым сегментам
median_segment_ratings = df_tv.groupby('audience_segment')['user rating score'].median().sort_values(ascending=False).reset_index()
median_segment_ratings.columns = ['Сегмент аудитории', 'Медианная оценка']

display(median_segment_ratings)

,Сегмент аудитории,Медианная оценка
0,Adults,89.0
1,Family,88.0
2,Teens,86.0
3,Kids,74.0


In [7]:
# Также построим BoxPlot по укрупненным сегментам аудитории (https://plotly.com/python/box-plots/)
fig_ratings = px.box(df_tv,
                     x='audience_segment',
                     y='user rating score',
                     color='audience_segment',
                     title='Распределение оценок ТВ-шоу по сегментам аудитории',
                     labels={'user rating score': 'Оценка пользователей', 'audience_segment': 'Сегмент аудитории'},)

fig_ratings.show()

**Промежуточный вывод:**

BoxPlot и медианы показывают, что оценки напрямую зависят от того, для кого снят сериал:

- Взрослые и семьи довольны больше всех: Топ по медианной оценке держит категория Adults (89.0 баллов). Буквально в шаге от нее идет семейный контент Family (88.0 баллов), а следом идут подростки Teens (86.0 баллов). Похоже, в этих жанрах Netflix умеет делать качественные вещи, которые точно попадают в ожидания зрителей.
- Детский сегмент проседает: У категории Kids заметный недобор по баллам. Медиана этого сегмента остановилась на отметке 74.0. Удержание детей у экрана через формат сериалов явно не в приоритете у платформы.

Техническая деталь: Все цифры посчитаны чисто по тем строкам, где изначально были оценки (dropna()). Никакого искусственного заполнения пустых клеток медианой по группам не проводилось, чтобы не двигать реальные границы распределения на графиках.